In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

In [2]:
def plot_loss_curves(history):
  plt.clf()
  history_dict = history.history
  loss_values = history_dict["loss"]
  val_loss_values = history_dict["val_loss"]
  epochs = range(1, len(loss_values) + 1)
  plt.plot(epochs, loss_values, "bo", label="Training loss")
  plt.plot(epochs, val_loss_values, "b", label="Validation loss")
  plt.title("Training and validation loss")
  plt.xlabel("Epochs")
  plt.ylabel("Loss")
  plt.legend()
  plt.show()

def plot_acc_curves(history):
  plt.clf()
  history_dict = history.history
  acc = history_dict["accuracy"]
  val_acc = history_dict["val_accuracy"]
  epochs = range(1, len(acc) + 1)
  plt.plot(epochs, acc, "bo", label="Training acc")
  plt.plot(epochs, val_acc, "b", label="Validation acc")
  plt.title("Training and validation accuracy")
  plt.xlabel("Epochs")
  plt.ylabel("Accuracy")
  plt.legend()
  plt.show()

## Retrieve data and preprocess

In [3]:
# Read data from URL

train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url,index_col=0)
val_df = pd.read_csv(val_url,index_col=0)
test_df = pd.read_csv(test_url,index_col=0)


print(f"""
Train samples: {train_df.shape[0]}
Validation samples: {val_df.shape[0]}
Test samples: {test_df.shape[0]}
""")


Train samples: 48991
Validation samples: 16331
Test samples: 21774



In [10]:
print(train_df.Lyric[:5])
print(train_df.Genre[:5])

0    Oh, girl. I can't get ready (Can't get ready f...
1    We met on a rainy evening in the summertime. D...
2    We carried you in our arms. On Independence Da...
3    I know he loved you. A long time ago. I ain't ...
4    Paralysis through analysis. Yellow moral uncle...
Name: Lyric, dtype: object
0     Pop
1     Pop
2    Rock
3     Pop
4    Rock
Name: Genre, dtype: object


In [7]:
# Let's one-hot-encode the target
y_train = pd.get_dummies(train_df['Genre']).to_numpy()
y_val = pd.get_dummies(val_df['Genre']).to_numpy()
y_test = pd.get_dummies(test_df['Genre']).to_numpy()

In [8]:
y_train[:10]

array([[False,  True, False],
       [False,  True, False],
       [False, False,  True],
       [False,  True, False],
       [False, False,  True],
       [False, False,  True],
       [False,  True, False],
       [False, False,  True],
       [False, False,  True],
       [ True, False, False]])

## Using pretrained embeddings

In [11]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

--2026-06-17 06:11:11--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-06-17 06:11:11--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-17 06:11:12--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [12]:
!unzip -q glove.6B.zip

In [13]:
! head -1 /content/glove.6B.100d.txt

the -0.038194 -0.24487 0.72812 -0.39961 0.083172 0.043953 -0.39141 0.3344 -0.57545 0.087459 0.28787 -0.06731 0.30906 -0.26384 -0.13231 -0.20757 0.33395 -0.33848 -0.31743 -0.48336 0.1464 -0.37304 0.34577 0.052041 0.44946 -0.46971 0.02628 -0.54155 -0.15518 -0.14107 -0.039722 0.28277 0.14393 0.23464 -0.31021 0.086173 0.20397 0.52624 0.17164 -0.082378 -0.71787 -0.41531 0.20335 -0.12763 0.41367 0.55187 0.57908 -0.33477 -0.36559 -0.54857 -0.062892 0.26584 0.30205 0.99775 -0.80481 -3.0243 0.01254 -0.36942 2.2167 0.72201 -0.24978 0.92136 0.034514 0.46745 1.1079 -0.19358 -0.074575 0.23353 -0.052062 -0.22044 0.057162 -0.15806 -0.30798 -0.41625 0.37972 0.15006 -0.53212 -0.2055 -1.2526 0.071624 0.70565 0.49744 -0.42063 0.26148 -1.538 -0.30223 -0.073438 -0.28312 0.37104 -0.25217 0.016215 -0.017099 -0.38984 0.87424 -0.72569 -0.51058 -0.52028 -0.1459 0.8278 0.27062


In [14]:
embedding_dim = 100
path_to_glove_file = f"glove.6B.{embedding_dim}d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [15]:
embeddings_index["movie"]

array([ 0.38251  ,  0.14821  ,  0.60601  , -0.51533  ,  0.43992  ,
        0.061053 , -0.62716  , -0.025385 ,  0.1643   , -0.22101  ,
        0.14423  , -0.37213  , -0.21683  , -0.08895  ,  0.097904 ,
        0.6561   ,  0.64455  ,  0.47698  ,  0.83849  ,  1.6486   ,
        0.88922  , -0.1181   , -0.012465 , -0.52082  ,  0.77854  ,
        0.48723  , -0.014991 , -0.14127  , -0.34747  , -0.29595  ,
        0.1028   ,  0.57191  , -0.045594 ,  0.026443 ,  0.53816  ,
        0.32257  ,  0.40788  , -0.043599 , -0.146    , -0.48346  ,
        0.32036  ,  0.55086  , -0.76259  ,  0.43269  ,  0.61753  ,
       -0.36503  , -0.60599  , -0.79615  ,  0.3929   , -0.23668  ,
       -0.34719  , -0.61201  ,  0.54747  ,  0.94812  ,  0.20941  ,
       -2.7771   , -0.6022   ,  0.8495   ,  1.2549   ,  0.017893 ,
       -0.041901 ,  2.1147   , -0.026618 , -0.28104  ,  0.68124  ,
       -0.14165  ,  0.99249  ,  0.49879  , -0.67538  ,  0.6417   ,
        0.42303  , -0.27913  ,  0.063403 ,  0.68909  , -0.3618

In [16]:
max_length = 300 # 90% of songs
max_tokens = 5000 # Use only common words for classification

text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode = "int",
    output_sequence_length=max_length,
)

In [17]:
text_vectorization.adapt(train_df['Lyric'])

In [21]:
print(len(text_vectorization.get_vocabulary()))
print(text_vectorization.get_vocabulary()[:10])

5000
['', '[UNK]', np.str_('the'), np.str_('you'), np.str_('i'), np.str_('to'), np.str_('and'), np.str_('a'), np.str_('me'), np.str_('it')]


In [24]:
text_vectorization(["Hey, how are you doing? Welcome to HODL"])

<tf.Tensor: shape=(1, 300), dtype=int64, numpy=
array([[134,  75,  58,   3, 445, 872,   5,   1,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
   

In [25]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [26]:
vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

counter = 0
embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    else:
        counter += 1

In [27]:
embedding_matrix.shape

(5000, 100)

In [32]:
print(embedding_matrix[0, :10])
print(embedding_matrix[1, :10])
print(embedding_matrix[2, :10])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[-0.038194   -0.24487001  0.72812003 -0.39961001  0.083172    0.043953
 -0.39140999  0.3344     -0.57545     0.087459  ]


In [33]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer= keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True
)

In [34]:
inputs = keras.Input(shape=(max_length,))
embedded = embedding_layer(inputs) # 300 x 100 table comes out
embedded = keras.layers.GlobalAveragePooling1D()(embedded) # 100-element vector
x = keras.layers.Dense(8, activation='relu')(embedded) # only hidden layer
outputs = keras.layers.Dense(3, activation="softmax")(x) # output layer

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 300, 100)  │    500,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 300)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 100)       │          0 │ embedding[0][0],  │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 8)         │        808 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │         27 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 500,835 (1.91 MB)

 Trainable params: 835 (3.26 KB)

 Non-trainable params: 500,000 (1.91 MB)

In [35]:
emb = model.layers[1]
emb.weights

[<Variable path=embedding/embeddings, shape=(5000, 100), dtype=float32, value=[[ 0.        0.        0.       ...  0.        0.        0.      ]
  [ 0.        0.        0.       ...  0.        0.        0.      ]
  [-0.038194 -0.24487   0.72812  ... -0.1459    0.8278    0.27062 ]
  ...
  [ 0.33947   0.16186   0.083984 ... -0.66934  -0.073064  0.91467 ]
  [-0.09169   0.10361   0.23232  ... -0.13697  -0.81738   0.3868  ]
  [ 0.28468  -0.26489   0.1649   ... -0.6158   -0.10085   1.0357  ]]>]

In [36]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [37]:
# Fit model

history = model.fit(x=X_train,
                    y=y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,
                    batch_size=32,)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.5487 - loss: 0.9270 - val_accuracy: 0.5556 - val_loss: 0.8888
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.5824 - loss: 0.8617 - val_accuracy: 0.5870 - val_loss: 0.8445
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.6043 - loss: 0.8304 - val_accuracy: 0.6047 - val_loss: 0.8219
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.6194 - loss: 0.8093 - val_accuracy: 0.6220 - val_loss: 0.8001
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6284 - loss: 0.7950 - val_accuracy: 0.6303 - val_loss: 0.7899
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.6329 - loss: 0.7871 - val_accuracy: 0.6334 - val_loss: 0.7829
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.6360 - loss: 0.7823 - val_accuracy: 0.6371 - val_loss: 0.7798
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.6388 - loss: 0.778

In [38]:
model.evaluate(x=X_test, y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6356 - loss: 0.7870


[0.7870206832885742, 0.6355745196342468]

In [39]:
emb = model.layers[1]
emb.weights

[<Variable path=embedding/embeddings, shape=(5000, 100), dtype=float32, value=[[ 0.        0.        0.       ...  0.        0.        0.      ]
  [ 0.        0.        0.       ...  0.        0.        0.      ]
  [-0.038194 -0.24487   0.72812  ... -0.1459    0.8278    0.27062 ]
  ...
  [ 0.33947   0.16186   0.083984 ... -0.66934  -0.073064  0.91467 ]
  [-0.09169   0.10361   0.23232  ... -0.13697  -0.81738   0.3868  ]
  [ 0.28468  -0.26489   0.1649   ... -0.6158   -0.10085   1.0357  ]]>]

## Fine-tuning pre-trined embeddings

In [40]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=True, # note that this has changed from before!!
    mask_zero=True
)

inputs = keras.Input(shape=(max_length,))
embedded = embedding_layer(inputs)
embedded = keras.layers.GlobalAveragePooling1D()(embedded)
x = keras.layers.Dense(8, activation='relu')(embedded)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 300, 100)  │    500,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 300)       │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 100)       │          0 │ embedding_1[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        808 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 8)         │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │         27 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 500,835 (1.91 MB)

 Trainable params: 500,835 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [42]:
# Fit model
history = model.fit(x=X_train, y=y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,
                    batch_size=32,)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - accuracy: 0.5737 - loss: 0.8857 - val_accuracy: 0.6447 - val_loss: 0.7352
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 39s 18ms/step - accuracy: 0.6157 - loss: 0.7997 - val_accuracy: 0.6746 - val_loss: 0.6997
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - accuracy: 0.6495 - loss: 0.7565 - val_accuracy: 0.7003 - val_loss: 0.6681
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 40s 21ms/step - accuracy: 0.6664 - loss: 0.7311 - val_accuracy: 0.7008 - val_loss: 0.6604
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.6708 - loss: 0.7223 - val_accuracy: 0.6988 - val_loss: 0.6602
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 43s 20ms/step - accuracy: 0.6739 - loss: 0.7175 - val_accuracy: 0.6974 - val_loss: 0.6598
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - accuracy: 0.6798 - loss: 0.7046 - val_accuracy: 0.7062 - val_loss: 0.6544
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 31s 20ms/step - accuracy: 0.6803 -

In [44]:
model.evaluate(x=X_test, y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7018 - loss: 0.6678


[0.6678073406219482, 0.7018002867698669]

In [45]:
emb = model.layers[1]
emb.weights

[<Variable path=embedding_1/embeddings, shape=(5000, 100), dtype=float32, value=[[ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.0837568  -0.15721107 -0.42420712 ...  0.24122941 -0.30150428
   -0.16348828]
  [-0.03577105 -0.2765272   0.5892187  ...  0.07322661  0.57869524
    0.048018  ]
  ...
  [ 0.01256885  0.2407912   0.4693873  ... -0.28208837 -0.17646205
    0.6278223 ]
  [ 0.71802485 -0.43829507  0.27781457 ... -0.35249782 -1.1700361
    1.1642704 ]
  [ 0.51025325 -0.36615548 -0.04785389 ... -0.74917686 -0.05481112
    1.2211237 ]]>]

## Learning from scratch

In [46]:
inputs = keras.Input(shape=(max_length,))

embedded = keras.layers.Embedding(input_dim=max_tokens,
                                  output_dim=64,
                                  mask_zero=True)(inputs)

embedded = keras.layers.GlobalAveragePooling1D()(embedded)

x = keras.layers.Dense(16, activation='relu')(embedded)

x = keras.layers.Dropout(0.5)(x)

outputs = keras.layers.Dense(3, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 300, 64)   │    320,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 300)       │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ embedding_2[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │      1,040 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 16)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 3)         │         51 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 321,091 (1.22 MB)

 Trainable params: 321,091 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [48]:
history = model.fit(x=X_train, y=y_train,
          validation_data=(X_val, y_val),
          epochs=10,
          batch_size=32,)

Epoch 1/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - accuracy: 0.6436 - loss: 0.7847 - val_accuracy: 0.7052 - val_loss: 0.6610
Epoch 2/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.7037 - loss: 0.6692 - val_accuracy: 0.7144 - val_loss: 0.6357
Epoch 3/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.7178 - loss: 0.6382 - val_accuracy: 0.7177 - val_loss: 0.6326
Epoch 4/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.7280 - loss: 0.6172 - val_accuracy: 0.7137 - val_loss: 0.6365
Epoch 5/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.7348 - loss: 0.6002 - val_accuracy: 0.7106 - val_loss: 0.6425
Epoch 6/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.7409 - loss: 0.5842 - val_accuracy: 0.7101 - val_loss: 0.6495
Epoch 7/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.7435 - loss: 0.5740 - val_accuracy: 0.7121 - val_loss: 0.6537
Epoch 8/10
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.7491 -

In [49]:
model.evaluate(x=X_test, y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7105 - loss: 0.7042


[0.7041762471199036, 0.7105262875556946]